# Notebook 2 - Chay thuat toan va so sanh

Doc cac JSON trong `data/`, chay lan luot cac thuat toan va luu ket qua vao `results/` trong repository. Moi lan chay duoc tach thanh process rieng voi timeout, nen thuat toan qua cham se bi dung ma khong lam crash kernel.


In [1]:
from pathlib import Path
import json
import multiprocessing as mp
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from min_cover_set import MinimumCoverSet, SetCoverInstance

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TIMEOUT_SECONDS = 120


In [ ]:
def load_instance(path):
    payload = json.loads(path.read_text(encoding="utf-8"))
    instance = SetCoverInstance.from_collections(payload["universe"], payload["subsets"])
    return instance, payload.get("metadata", {})

def run_algorithm_in_child(instance, method_name, queue):
    try:
        selected = getattr(MinimumCoverSet(instance), method_name)()
        queue.put({"status": "completed", "selected": len(selected), "solution": list(selected)})
    except Exception as error:
        queue.put({"status": "error", "error": f"{type(error).__name__}: {error}"})

def run_with_timeout(instance, method_name, timeout_seconds=TIMEOUT_SECONDS):
    context = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
    queue = context.Queue()
    process = context.Process(target=run_algorithm_in_child, args=(instance, method_name, queue))
    started = time.perf_counter()
    process.start()
    process.join(timeout_seconds)
    elapsed_ms = (time.perf_counter() - started) * 1000
    if process.is_alive():
        process.kill()
        process.join()
        return {"status": "timeout", "runtime_ms": elapsed_ms, "selected": None, "solution": None}
    if queue.empty():
        return {"status": "error", "runtime_ms": elapsed_ms, "selected": None, "solution": None, "error": "child process exited without a result"}
    result = queue.get()
    result["runtime_ms"] = elapsed_ms
    result.setdefault("selected", None)
    result.setdefault("solution", None)
    return result

records = []
algorithm_names = {
    "greedy": "greedy",
    "brute_force": "brute_force",
    "dynamic_programming": "dynamic_programming",
    "branch_and_bound": "branch_and_bound",
}

for path in sorted(DATA_DIR.glob("n*.json")):
    instance, metadata = load_instance(path)
    optimal_size = None
    for name, method_name in algorithm_names.items():
        result = run_with_timeout(instance, method_name)
        if name == "brute_force" and result["status"] == "completed":
            optimal_size = result["selected"]
        result.update({"file": path.name, **metadata, "algorithm": name, "optimal_size": optimal_size})
        if result["selected"] is not None and optimal_size:
            result["accuracy"] = optimal_size / result["selected"]
            result["is_optimal"] = result["selected"] == optimal_size
        else:
            result["accuracy"] = None
            result["is_optimal"] = None
        records.append(result)

results_df = pd.DataFrame(records)
results_df.to_json(RESULTS_DIR / "results.json", orient="records", indent=2)
results_df.to_csv(RESULTS_DIR / "results.csv", index=False)
print(f"Da luu {len(results_df)} dong ket qua vao {RESULTS_DIR}")
print(results_df["status"].value_counts())
results_df.head()


In [ ]:
completed = results_df[results_df["status"] == "completed"]
summary = (
    completed.groupby(["num_elements", "density", "algorithm"], as_index=False)
    .agg(runtime_ms=("runtime_ms", "mean"), accuracy=("accuracy", "mean"), selected=("selected", "mean"))
)
display(summary.round({"runtime_ms": 3, "accuracy": 3, "selected": 2}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for algorithm, group in results_df.groupby("algorithm"):
    completed_group = group[group["status"] == "completed"]
    runtime = completed_group.groupby("num_elements", as_index=False)["runtime_ms"].mean()
    axes[0].plot(runtime["num_elements"], runtime["runtime_ms"], marker="o", label=algorithm)
    quality = completed_group.groupby("num_elements", as_index=False)["accuracy"].mean()
    axes[1].plot(quality["num_elements"], quality["accuracy"], marker="o", label=algorithm)
axes[0].set(title="Runtime theo kich thuoc", xlabel="So phan tu", ylabel="Milliseconds")
axes[1].set(title="Accuracy trung binh", xlabel="So phan tu", ylabel="Optimal size / selected size")
for axis in axes:
    axis.grid(True, alpha=0.3)
    axis.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "comparison.png", dpi=150)
plt.show()
print(f"Da luu bieu do: {RESULTS_DIR / 'comparison.png'}")
